In [1]:
import numpy as np

data_fp = "../../data/mces-data/mces_distances.npz"
with np.load(data_fp, allow_pickle=True) as mces_data:
    # Show the keys (column names) available in the loaded npz file
    print("Available keys (column names):", mces_data.files)
    # Show the shape of each array in the npz file
    print("Shape of 'mces' array:", mces_data['mces'].shape)
    print("Shape of 'smiles' array:", mces_data['smiles'].shape)
    mces_values = mces_data['mces'].copy()
    smiles_values = mces_data['smiles'].copy()

Available keys (column names): ['mces', 'smiles']
Shape of 'mces' array: (2012094766,)
Shape of 'smiles' array: (63437,)


In [11]:
print("\nFirst 10 rows of mces array:")
print(mces_values[:10])


First 10 rows of mces array:
[-1. -1. -1. 27. -1. -1. -1. 26. -1. -1.]


In [12]:
#Show first 10 entries of the 'smiles' array
#print("\nFirst 9 entries of smiles array:")
print(smiles_values[:9])


['[2H]C1(C([N+]2=CC=CC=C2C3=CC=CC=[N+]31)([2H])[2H])[2H].[Br-].[Br-]'
 '[2H]C([2H])([2H])[C@@H](CC1=CC=CC=C1)N.[2H]C([2H])([2H])[C@@H](CC1=CC=CC=C1)N'
 '[2H]C([2H])([2H])[C@@H](CC1=CC=CC=C1)N.[2H]C([2H])([2H])[C@@H](CC1=CC=CC=C1)N.OS(=O)(=O)O'
 '[As]#[In]' 'B1(C2CCCC1CCC2)B3C4CCCC3CCC4'
 'B1(C2=C(C=C(C=C2)Cl)C=NN1S(=O)(=O)CCC)O'
 'B1(C2=C(C=C(C=C2)CNC(=O)C3=C(C=C(C=C3)C4=NOC(C4)(C5=CC(=C(C(=C5)Cl)F)Cl)C(F)(F)F)C)C(O1)(C)C)O'
 'B1(C2=CC=CC=C2C=NN1S(=O)(=O)C3=CC=C(C=C3)C)O'
 '[B-]1(=C2C=C(C=CC2=CO1)OC3=C(C=C(C=C3)CN)F)O']


In [6]:
from scipy.spatial.distance import squareform
mces_values_matrix = squareform(mces_values)
mces_values_matrix[0,0:10]

array([ 0., -1., -1., -1., 27., -1., -1., -1., 26., -1.], dtype=float16)

In [ ]:
from myopic_mces import MCES

#smiles_l = mces_data['smiles'].copy().tolist()  # Convert to list of bytes
num_smiles = len(smiles_values)

solver_options={}
solver_options['msg'] = False

mces_idx = 0
for i in range(0, 2):
    for j in range(i + 1, num_smiles):
        #MCES(smiles_values[i], smiles_values[j])
        if mces_values[mces_idx] > 0:
            mces = MCES(smiles_values[i], smiles_values[j], solver_options = solver_options)
            if mces[1] != mces_values[mces_idx]:
                print(f"Mismatch for {smiles_values[i]} and {smiles_values[j]}: computed {mces[1]}, expected {mces_values[mces_idx]}")
            #print(f"MCES between {smiles_values[i]} and {smiles_values[j]}: {mces[1]},  mces_values[{mces_idx}] = {mces_values[mces_idx]}")
        mces_idx += 1

In [24]:
from numpy import triu_indices, sqrt
n = 10
for k in range(int(n*(n-1)/2)):
    i = n - 2 - int(sqrt(-8*k + 4*n*(n-1)-7)/2.0 - 0.5)
    j = k + i + 1 - n*(n-1)/2 + (n-i)*((n-i)-1)/2
    assert np.triu_indices(n, k=1)[0][k] == i
    assert np.triu_indices(n, k=1)[1][k] == j

for i in range(n):
    for j in range(i+1, n):
        k = int(n*(n-1)/2) - int((n-i)*((n-i)-1)/2) + j - i - 1
        #print(k)
        assert triu_indices(n, k=1)[0][k] == i
        assert triu_indices(n, k=1)[1][k] == j

In [25]:
# Get unique values from the mces array
unique_mces = np.unique(mces_data['mces'])
print("Unique values in mces array:")
print(unique_mces)
print(f"Number of unique values: {len(unique_mces)}")

Unique values in mces array:
[ -1.   0.   1. ... 649. 651. 674.]
Number of unique values: 1268


In [26]:
# Check if all values in unique_mces are integers
all_ints = np.all(unique_mces == np.floor(unique_mces))
print(f"Are all values in unique_mces integers? {all_ints}")

# Alternative check - see if any values have decimal parts
has_decimals = np.any(unique_mces % 1 != 0)
print(f"Do any values have decimal parts? {has_decimals}")

# Show some examples of the values
print(f"First 10 unique values: {unique_mces[:10]}")
print(f"Last 10 unique values: {unique_mces[-10:]}")

Are all values in unique_mces integers? False
Do any values have decimal parts? True
First 10 unique values: [-1.   0.   1.   1.5  2.   2.5  3.   3.5  4.   4.5]
Last 10 unique values: [639.  643.5 644.  645.  645.5 647.  648.  649.  651.  674. ]


In [27]:
# Count how many pairs are not -1
not_minus_one_count = np.sum(mces_data['mces'] != -1)
total_count = len(mces_data['mces'])

print(f"Number of pairs that are not -1: {not_minus_one_count}")
print(f"Total number of pairs: {total_count}")
print(f"Percentage of pairs that are not -1: {(not_minus_one_count / total_count) * 100:.2f}%")

Number of pairs that are not -1: 1239677623
Total number of pairs: 2012094766
Percentage of pairs that are not -1: 61.61%
